# 使用 Codex CLI 自动修复 GitHub 上的 CI 失败

> **已归档示例——请使用当前的 CI 指南。** 下面的工作流会将 API 密钥暴露给整个 job，并将仓库控制的执行与仓库写权限组合在一起。请不要照搬这种凭据和权限布局。请使用[当前的自动修复工作流](https://learn.chatgpt.com/docs/non-interactive-mode#example-autofix-ci-failures-in-github-actions)（它将补丁生成与 pull request 发布分离），并参考 [Codex GitHub Action 安全检查清单](https://learn.chatgpt.com/docs/github-action#security-checklist)。

## 本 cookbook 的目的

本 cookbook 展示如何将 OpenAI Codex CLI 嵌入到你的 CI/CD 流水线中，以便在构建或测试失败时，codex 能自动生成并提出修复方案。下面是一个在 GitHub Actions 中运行 CI 的 node 项目示例。

## 端到端流程

下面是我们将要实现的流水线流程：


<img src="../../images/ci-codex-workflow.png"  width="700"/>

## 前提条件

- 一个带有 Actions 工作流的 GitHub 仓库

- 你需要在 GitHub 设置中创建 `OPENAI_API_KEY` 环境变量，位置为 https://github.com/{org-name}/{repo-name}/settings/secrets/actions. 你也可以在组织级别设置它（以便在多个仓库之间共享 secrets）

- Codex 需要预先安装 python 才能使用 `codex login`

- 你需要在仓库以及组织中勾选允许 actions 创建 PR 的设置：


<img src="../../images/github-pr-settings.png" width="700"/>



## 第 1 步：将 Github Action 添加到你的 CI 流水线

下面的 YAML 展示了一个 GitHub action：它会在 CI 失败时自动触发，安装 Codex，使用 codex exec，然后在失败的分支上创建一个包含修复的 PR。请将 "CI" 替换为你想要监控的工作流名称。

```yaml
name: Codex Auto-Fix on Failure

on:
  workflow_run:
    # Trigger this job after any run of the primary CI workflow completes
    workflows: ["CI"]
    types: [completed]

permissions:
  contents: write
  pull-requests: write

jobs:
  auto-fix:
    # Only run when the referenced workflow concluded with a failure
    if: ${{ github.event.workflow_run.conclusion == 'failure' }}
    runs-on: ubuntu-latest
    env:
      OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
      FAILED_WORKFLOW_NAME: ${{ github.event.workflow_run.name }}
      FAILED_RUN_URL: ${{ github.event.workflow_run.html_url }}
      FAILED_HEAD_BRANCH: ${{ github.event.workflow_run.head_branch }}
      FAILED_HEAD_SHA: ${{ github.event.workflow_run.head_sha }}
    steps:
      - name: Check OpenAI API Key Set
        run: |
          if [ -z "$OPENAI_API_KEY" ]; then
            echo "OPENAI_API_KEY secret is not set. Skipping auto-fix." >&2
            exit 1
          fi
      - name: Checkout Failing Ref
        uses: actions/checkout@v4
        with:
          ref: ${{ env.FAILED_HEAD_SHA }}
          fetch-depth: 0

      - name: Setup Node.js
        uses: actions/setup-node@v4
        with:
          node-version: '20'
          cache: 'npm'

      - name: Install dependencies
        run: |
          if [ -f package-lock.json ]; then npm ci; else npm i; fi
      - name: Run Codex
        uses: openai/codex-action@main
        id: codex
        with:
          openai_api_key: ${{ secrets.OPENAI_API_KEY }}
          prompt: "You are working in a Node.js monorepo with Jest tests and GitHub Actions. Read the repository, run the test suite, identify the minimal change needed to make all tests pass, implement only that change, and stop. Do not refactor unrelated code or files. Keep changes small and surgical."
          codex_args: '["--config","sandbox_mode=\"workspace-write\""]'

      - name: Verify tests
        run: npm test --silent

      - name: Create pull request with fixes
        if: success()
        uses: peter-evans/create-pull-request@v6
        with:
          commit-message: "fix(ci): auto-fix failing tests via Codex"
          branch: codex/auto-fix-${{ github.event.workflow_run.run_id }}
          base: ${{ env.FAILED_HEAD_BRANCH }}
          title: "Auto-fix failing CI via Codex"
          body: |
            Codex automatically generated this PR in response to a CI failure on workflow `${{ env.FAILED_WORKFLOW_NAME }}`.
            Failed run: ${{ env.FAILED_RUN_URL }}
            Head branch: `${{ env.FAILED_HEAD_BRANCH }}`
            This PR contains minimal changes intended solely to make the CI pass.
```


## 第 2 步：Actions 工作流启动

你可以进入仓库下的 Actions 标签页，查看 Actions 工作流中失败的 job。

<img src="../../images/failing-workflow.png" width="700"/>


失败的工作流完成后，应会触发 Codex 工作流。


<img src="../../images/codex-workflow.png" width="700"/>



## 第 3 步：确认 Codex 已创建待审查的 PR
Codex 工作流执行完成后，它应该会从功能分支 codex/auto-fix 发起一个 pull request。检查一切是否正常，然后将其合并。

<img src="../../images/codex-pr.png" width="700"/>



## 结论

这一自动化方案将 OpenAI Codex CLI 与 GitHub Actions 无缝集成，能够针对失败的 CI 运行自动提出修复方案。

借助 Codex，你可以减少人工干预、加快代码审查，并保持主分支的健康。该工作流确保测试失败能够被快速高效地处理，让开发者专注于更高价值的任务。在[这里](https://github.com/openai/codex/)进一步了解 codex-cli 及其功能。